# The columns we added, and why

FMA ships 518 pre-computed audio features. This notebook is about the ones it
does not ship.

The listening test asks raters two things about every pair: how alike the tracks
sound, and — from a fixed vocabulary — what connects them. The second answer is
the one that turns into work. Counting the words and lining them up against the
feature families the model actually has gives a short list of things people keep
hearing that the model cannot see.

Two words came back with nothing behind them:

| word | mentions | mean score | families in FMA |
|---|---:|---:|---|
| `rhythm` | 16 | 4.12 | none |
| `instrumentation` | 7 | 4.29 | none |

Both were extracted from the mp3s. A feature that is not in the feature table
cannot be recovered from it — you have to go back to the audio.

This notebook walks through what was extracted, column by column, and which of
it survived. `03_listening_test_results.ipynb` has the measurements that decide
whether any of it helped.

## First: confirming the hole

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

META = "../data/fma_metadata"
tracks = pd.read_csv(f"{META}/tracks.csv", index_col=0, header=[0, 1])
small = tracks[tracks[("set", "subset")] == "small"]
X = pd.read_csv(f"{META}/features.csv", index_col=0, header=[0, 1, 2]).loc[small.index].dropna()

fma = pd.DataFrame({"columns": X.columns.get_level_values(0).value_counts()})
fma["share"] = (fma["columns"] / fma["columns"].sum()).map("{:.0%}".format)
fma["measures"] = [
    {"chroma_cens": "pitch", "chroma_cqt": "pitch", "chroma_stft": "pitch",
     "tonnetz": "pitch", "mfcc": "texture", "spectral_contrast": "texture",
     "spectral_centroid": "texture", "spectral_bandwidth": "texture",
     "spectral_rolloff": "texture", "zcr": "texture", "rmse": "energy"}[f]
    for f in fma.index]
fma

,columns,share,measures
feature,,,
mfcc,140,27%,texture
chroma_cens,84,16%,pitch
chroma_cqt,84,16%,pitch
chroma_stft,84,16%,pitch
spectral_contrast,49,9%,texture
tonnetz,42,8%,pitch
rmse,7,1%,energy
spectral_bandwidth,7,1%,texture
spectral_centroid,7,1%,texture


Eleven families, 518 columns, and every one of them is pitch, texture or energy.
Nothing measures time: not tempo, not pulse, not how many notes go by in a
second. Nothing measures what is producing the sound either — a solo guitar and
a drum loop can sit close on all eleven.

So the two words the raters used most were both invisible to the model.

## The rhythm columns

`modeling/rhythm.py` reads each mp3 and builds one signal first: the **onset
envelope**. It peaks wherever a note or a hit begins, and is flat in between —
a map of where the events are in a track.

Eleven of the thirteen columns are readings of that one curve.

In [2]:
from modeling.rhythm import load as load_rhythm

R = load_rhythm()
described = {
    ("tempo", "mean"):             "beats per minute, from the envelope's dominant period",
    ("onset_rate", "mean"):        "note onsets per second — density, not speed",
    ("onset_strength", "mean"):    "how hard the average event hits",
    ("onset_strength", "median"):  "the same, robust to a few loud hits",
    ("onset_strength", "std"):     "how much the hits vary — even or dynamic",
    ("onset_strength", "skew"):    "mostly quiet with occasional sharp hits?",
    ("onset_strength", "kurtosis"):"a few extreme transients, like a dry snare",
    ("onset_strength", "max"):     "the single hardest hit in the track",
    ("pulse_clarity", "mean"):     "how firmly the track holds its beat",
    ("pulse_clarity", "median"):   "the same, robust",
    ("pulse_clarity", "std"):      "how much its grip on the beat wanders",
    ("pulse_clarity", "skew"):     "does the pulse come in bursts?",
    ("pulse_clarity", "kurtosis"): "a few moments of very strong pulse",
}
pd.DataFrame({"what it measures": [described[(a, b)] for a, b, _ in R.columns]},
             index=pd.MultiIndex.from_tuples([(a, b) for a, b, _ in R.columns],
                                             names=["family", "statistic"]))

what it measures
family         statistic                                                   
tempo          mean       beats per minute, from the envelope's dominant...
onset_rate     mean             note onsets per second — density, not speed
onset_strength mean                         how hard the average event hits
               std                 how much the hits vary — even or dynamic
               skew                mostly quiet with occasional sharp hits?
               kurtosis          a few extreme transients, like a dry snare
               median                   the same, robust to a few loud hits
               max                      the single hardest hit in the track
pulse_clarity  mean                     how firmly the track holds its beat
               std                    how much its grip on the beat wanders
               skew                          does the pulse come in bursts?
               kurtosis                  a few moments of very strong pulse
               median                                      the same, robust

`pulse_clarity` needs a word. librosa's predominant-local-pulse curve tracks how
strongly a track is keeping the beat at each moment — metronomic high, rubato
low. The intention was to separate *same tempo* from *same groove*.

Its `min` and `max` were dropped at extraction: librosa normalises the curve to
[0, 1] per track, so those two are the same number for all 8,000 and carry
nothing. A constant column also makes StandardScaler divide by zero.

### What each column actually separates

In [3]:
genre = small[("track", "genre_top")].reindex(R.index)
flat = R.copy()
flat.columns = [f"{a}.{b}" for a, b, _ in R.columns]

by_genre = flat.groupby(genre).mean()
by_genre[["tempo.mean", "onset_rate.mean", "onset_strength.mean",
          "onset_strength.kurtosis", "pulse_clarity.mean"]].round(2)

,tempo.mean,onset_rate.mean,onset_strength.mean,onset_strength.kurtosis,pulse_clarity.mean
"(track, genre_top)",,,,,
Electronic,122.48,4.66,1.61,22.84,0.22
Experimental,122.05,3.30,1.25,77.81,0.19
Folk,121.41,3.61,1.22,24.00,0.21
Hip-Hop,115.46,4.53,1.73,11.89,0.23
Instrumental,120.25,3.04,1.12,54.82,0.21
International,125.22,4.45,1.56,12.83,0.22
Pop,122.24,3.76,1.34,23.32,0.22
Rock,125.25,3.20,1.16,57.49,0.21


Read across the rows rather than down.

`onset_rate` separates cleanly: Electronic and Hip-Hop run at 4.5–4.7 events a
second, Instrumental and Rock at 3.0–3.2. `onset_strength.mean` does too —
Hip-Hop 1.73 against Instrumental 1.12.

`tempo` does not. Every genre averages between 115 and 125 bpm.

`pulse_clarity` barely moves either: 0.19 to 0.23 across all eight.

### Putting a number on that

`eta²` is the share of a column's spread that genre explains. Zero means knowing
the genre tells you nothing about the column; one means it tells you exactly.
It is cheap enough to run on a column before deciding whether it is worth
keeping.

In [4]:
def eta2(col, by):
    """Share of a column's variance that the grouping explains."""
    total = ((col - col.mean()) ** 2).sum()
    within = sum(((g - g.mean()) ** 2).sum() for _, g in col.groupby(by))
    return 1 - within / total


scores = pd.DataFrame({
    "eta2": {c: eta2(flat[c], genre) for c in flat.columns}
}).sort_values("eta2", ascending=False).round(3)
scores["verdict"] = np.where(scores["eta2"] > 0.10, "carries something",
                    np.where(scores["eta2"] > 0.05, "weak", "近 nothing"))
scores["verdict"] = scores["verdict"].str.replace("近 ", "", regex=False)
scores

,eta2,verdict
onset_strength.mean,0.283,carries something
onset_strength.median,0.234,carries something
onset_strength.std,0.177,carries something
onset_rate.mean,0.150,carries something
onset_strength.skew,0.100,weak
onset_strength.max,0.081,weak
pulse_clarity.std,0.079,weak
onset_strength.kurtosis,0.075,weak
pulse_clarity.mean,0.071,weak
pulse_clarity.skew,0.059,weak


The four columns that carry the family are `onset_strength.mean`, its `median`
and `std`, and `onset_rate` — all readings of how hard and how often a track
hits. Tempo comes last at 0.019, below every other column in the block.

Two reasons tempo fails here. Estimated tempo has a well-known octave
ambiguity — a 70 bpm track is often read as 140 — and even correct, speed is not
what distinguishes a genre. Folk at 120 and Electronic at 120 are not alike.

The five `pulse_clarity` columns are also nearly interchangeable with each
other, correlating 0.95 to 0.98. They were kept because they cost nothing to
carry, but they are not doing the work.

## The harmonic–percussive columns

The second hole was `instrumentation` — nothing in the feature set says whether
a track is carried by drums or by instruments.

`modeling/separation.py` splits the audio on a geometric fact about
spectrograms. A sustained note holds one frequency over time and draws a
horizontal line; a hit covers every frequency for an instant and draws a
vertical one. Median-filtering along each axis separates them into two signals:
the track without drums, and the drums alone.

Three families come out of it.

In [5]:
from modeling.separation import load as load_sep

S = load_sep()
pd.DataFrame({
    "columns": S.columns.get_level_values(0).value_counts(),
    "what it is": {
        "hpss_ratio": "log energy ratio, percussive over harmonic",
        "mfcc_harmonic": "the timbre of the instruments alone (20 x mean, std)",
        "mfcc_percussive": "the timbre of the drums alone (20 x mean, std)",
    },
})

,columns,what it is
hpss_ratio,1,"log energy ratio, percussive over harmonic"
mfcc_harmonic,40,the timbre of the instruments alone (20 x mean...
mfcc_percussive,40,"the timbre of the drums alone (20 x mean, std)"


### The ratio, by genre

In [6]:
ratio = S[("hpss_ratio", "mean", "01")]
g2 = small[("track", "genre_top")].reindex(S.index)
pd.DataFrame({
    "log ratio": ratio.groupby(g2).mean().round(2),
    "percussive / harmonic": np.exp(ratio.groupby(g2).mean()).round(2),
}).sort_values("log ratio", ascending=False)

,log ratio,percussive / harmonic
"(track, genre_top)",,
Hip-Hop,-0.69,0.50
Rock,-1.41,0.25
Electronic,-1.42,0.24
International,-1.44,0.24
Pop,-1.44,0.24
Experimental,-1.72,0.18
Folk,-2.32,0.10
Instrumental,-2.60,0.07


Electronic and Experimental carry several times more percussive than harmonic
energy. Folk and Instrumental carry a fraction. That is a seven-fold spread
between the ends, from one column, describing something no other column in the
project describes.

### One transform that decided whether this column worked

Measured raw, the ratio scored `eta² = 0.038` — near useless. The same column
after a log scored `0.143`.

A ratio is multiplicative: four times more percussive and four times less are
the same distance from even, but only on a log scale. Raw, the tail runs past 20
and a handful of extreme tracks swallow the variance.

`separation.py` takes the log at extraction, so nothing downstream has to
remember to.

In [7]:
raw = np.exp(ratio)
pd.DataFrame({
    "eta2": [round(eta2(raw, g2), 3), round(eta2(ratio, g2), 3)],
    "min": [round(raw.min(), 2), round(ratio.min(), 2)],
    "max": [round(raw.max(), 2), round(ratio.max(), 2)],
}, index=["raw ratio", "log ratio"])

,eta2,min,max
raw ratio,0.058,0.00,26.89
log ratio,0.224,-10.52,3.29


## What survived

Of the 94 columns extracted across both modules, **14 are in the model the
recommender now runs on** — `modeling.features.FEATURE_SETS["heard"]`, which is
mfcc plus these fourteen.

The 80 split-mfcc columns were dropped. They lifted genre agreement to the best
number in the whole project, 0.488, and left the supervised ceiling flat, which
means they restated what the unsplit mfcc already said rather than adding to it.
They also made the correlation with human ratings worse. Judged on agreement
alone we would have kept them.

In [8]:
pd.DataFrame([
    ("rhythm",     "tempo, onset_rate, onset_strength, pulse_clarity", 13, "kept",
     "ceiling 0.485 -> 0.526, and R2 0.165 against timbre: genuinely new"),
    ("separation", "hpss_ratio",                                        1, "kept",
     "ceiling 0.526 -> 0.533, and it improves the human correlation"),
    ("separation", "mfcc_harmonic, mfcc_percussive",                   80, "dropped",
     "best genre score in the project, ceiling flat 0.533 -> 0.534: nothing new"),
], columns=["module", "families", "columns", "verdict", "why"]).set_index("module")

,families,columns,verdict,why
module,,,,
rhythm,"tempo, onset_rate, onset_strength, pulse_clarity",13,kept,"ceiling 0.512 -> 0.546, and R2 0.165 against t..."
separation,hpss_ratio,1,kept,"ceiling 0.528 -> 0.536, and it improves the hu..."
separation,"mfcc_harmonic, mfcc_percussive",80,dropped,"best genre score in the project, ceiling flat,..."


### Rebuilding either file

Both are written straight to `data/`, which is gitignored, so they have to be
rebuilt in a fresh checkout. Rhythm takes about 90 seconds on eight cores; the
split is the slow one at roughly 20 minutes, because separating the audio costs
more than reading it.

```bash
uv run python -m modeling.rhythm
uv run python -m modeling.separation
```

Seven tracks fail in both: six are truncated files FMA is known to ship, and one
is silent, which leaves the onset envelope flat and its skew undefined. They are
dropped rather than filled, the same way `load_features` drops rows with NaNs —
a partially described track has no business in a distance calculation.

### The word still without a feature

`mood` was the most used word of all, 17 times, on pairs averaging 4.00. Nothing
was built for it, and probably nothing should be: mood reads as a combination of
energy, harmony and tempo rather than a measurable thing of its own. If the
remaining four raters keep naming it, the question is worth reopening — but as a
derived reading of the columns above, not as a new extraction.